# VNetra-Lite — Analisis Kuantitatif Data Sesi Pengujian

**Program Studi Teknik Elektro | Skripsi S1**

---

## Tentang Notebook

Notebook ini adalah instrumen analisis kuantitatif untuk memvalidasi kinerja sistem *Electronic Travel Aid* (ETA) VNetra-Lite.
Data diperoleh dari SessionDataLogger.kt yang merekam setiap siklus valuateObstacles() selama sesi pengujian lapangan.

| Figur | Judul | Tujuan Validasi |
|:---:|:---|:---|
| **Fig. 4.1** | Threshold Adaptif vs Waktu | Respons dinamis algoritma terhadap kecepatan |
| **Fig. 4.2** | Kecepatan vs Threshold | Validasi formula *tanh* — Pers. 3.9 BAB III |
| **Fig. 4.3** | Dekomposisi Latensi *End-to-End* | Analisis bottleneck komunikasi per komponen |
| **Fig. 4.4** | Packet Loss dan PDR | Reliabilitas transmisi UDP (Pers. 2.13) |
| **Tabel 4.x** | Ringkasan Statistik | Metrik kuantitatif sesi pengujian lengkap |
| **Fig. 4.5** | *Confusion Matrix* Akurasi Arah | Presisi/Recall deteksi 6 kelas arah jam |

**Cara Penggunaan:**
1. Salin VNetra_Session_*.csv dari Android ke folder nalysis/ ini.
2. Sesuaikan nilai di **Sel Konfigurasi** jika diperlukan.
3. Untuk data nyata: *comment* baris df = generate_dummy(), *uncomment* Sel Load CSV.
4. Jalankan semua sel **secara berurutan** (▶▶ *Run All*). Gambar tersimpan di output/ (300 DPI).


In [ ]:
# !pip install pandas matplotlib numpy ipywidgets
import os, glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
from matplotlib.lines import Line2D

# ── Gaya plot: Q1 Journal Standard (IEEE / Elsevier) ──────────────────────
# Mengacu pada IEEE Author Guidelines & Elsevier Artwork Instructions:
# - Latar putih, teks hitam, sans-serif (Arial equiv.), minimal ink.
# - Grid tipis abu-abu, tanpa frame tebal, legend bersih.
# - Resolusi cetak 300 DPI, ukuran font 9-10 pt.
plt.rcParams.update({
    # Canvas
    'figure.facecolor':   'white',
    'axes.facecolor':     'white',
    'axes.edgecolor':     '#333333',
    'axes.linewidth':     0.8,
    # Text
    'text.color':         '#111111',
    'axes.labelcolor':    '#111111',
    'xtick.color':        '#333333',
    'ytick.color':        '#333333',
    'axes.labelsize':     10,
    'xtick.labelsize':    9,
    'ytick.labelsize':    9,
    'axes.titlesize':     10,
    'axes.titleweight':   'bold',
    # Font — sans-serif mirip Arial (IEEE standard)
    'font.family':        'sans-serif',
    'font.sans-serif':    ['DejaVu Sans', 'Arial', 'Helvetica'],
    # Grid — tipis, tidak mengganggu data
    'axes.grid':          True,
    'grid.color':         '#CCCCCC',
    'grid.linewidth':     0.5,
    'grid.linestyle':     '--',
    'grid.alpha':         0.7,
    # Tick
    'xtick.direction':    'in',
    'ytick.direction':    'in',
    'xtick.major.size':   3.5,
    'ytick.major.size':   3.5,
    # Legend
    'legend.facecolor':   'white',
    'legend.edgecolor':   '#AAAAAA',
    'legend.fontsize':    9,
    'legend.framealpha':  0.9,
    # Savefig
    'figure.dpi':         100,
    'savefig.dpi':        300,
    'savefig.bbox':       'tight',
    'savefig.facecolor':  'white',
})
OUTPUT_DIR = 'output'  # direktori output untuk semua figur
os.makedirs(OUTPUT_DIR, exist_ok=True)

# ── Palet warna Q1 Journal (color-blind safe, WCAG-compliant) ───────────────
# Menggunakan subset dari Wong (2011) color-blind safe palette:
# Nature Methods 8(6):441 — direkomendasikan untuk publikasi ilmiah.
C_THRESHOLD = '#0072B2'   # Biru tua    — Threshold T (IBM Blue)
C_DISTANCE  = '#D55E00'   # Oranye tua  — Jarak objek (vermillion)
C_THEORY    = '#E69F00'   # Oranye muda — Kurva teoritis (orange)
C_ALERT     = '#CC79A7'   # Ungu muda   — Momen alert
C_SAFE      = '#009E73'   # Hijau tua   — d0 (bluish green)
C_MAX       = '#D62728'   # Merah       — T_max
C_SCATTER   = '#0072B2'   # Biru        — Scatter data lapangan
print('Lingkungan analisis Q1-journal siap.')

---
## Sel 2 — Konfigurasi Parameter Formula

Semua konstanta di bawah ini **harus konsisten** dengan `VNetraConfig.kt` dan nilai yang didokumentasikan di **Bab 3**.

| Parameter | Simbol | Nilai | Sumber Literatur |
|:---|:---:|:---:|:---|
| Jarak aman minimum | $d_0$ | 1000 mm | Desain sistem (1 m konservatif) |
| Batas threshold maksimum | $T_{max}$ | 4000 mm | Jangkauan valid VL53L5CX |
| Perception-Reaction Time | $t_R$ | 1.3 s | Kovacs & Nagy (2020) [44] |
| Durasi langkah penuh | $t_{step}$ | 0.63 s | Knoblauch et al. [42]; Winter [45] |

In [ ]:
# Sesuaikan dengan nilai di VNetraConfig.kt Anda
D0     = 1000    # BASE_WARNING_DIST_MM (mm)
T_MAX  = 4000    # MAX_THRESHOLD_MM (mm)
T_R    = 1.3     # PERCEPTION_REACTION_TIME_SEC — Kovacs & Nagy [44]
T_STEP = 0.632   # STEP_DURATION_SEC — Knoblauch et al. [42]; Winter [45]
print(f'Config: d0={D0}mm | T_max={T_MAX}mm | t_R={T_R}s | t_step={T_STEP}s')

---
## Sel 3 — Generator Data Dummy

> **Catatan:** Blok ini menghasilkan data **simulasi** untuk keperluan pengembangan dan verifikasi logika notebook tanpa file CSV lapangan. Setelah pengujian lapangan selesai, **ganti dengan Sel 4 (Load CSV)** di bawahnya.

Generator menggunakan **siklus periodik** (±120 s/siklus) sehingga realistis untuk durasi berapa pun (60 s, 600 s, hingga 3600 s).

Setiap siklus terdiri dari empat fase:
1. **Idle** (10 s) — Sistem diam, halangan jauh (±4500 mm)
2. **Approach** (60 s) — Berjalan mendekati halangan,  ≈$ 1.0–1.3 m/s
3. **Alert** (30 s) — Halangan dalam jangkauan, sistem memperingatkan
4. **Recover** (20 s) — Pengguna menghindari, halangan kembali menjauh

| Kolom CSV | Tipe | Penjelasan |
|:---|:---:|:---|
| lapsed_s | float | Waktu sejak awal sesi (detik) |
| d_obj_mm | int | Jarak objek terdekat (mm) |
| 	hreshold_T_mm | int | Threshold dinamis $ (mm) |
| _avg_mmps | float | Kecepatan EWMA (mm/s) |
| m_buffer_mm | float | Jarak pengereman $= 0.5 |a| t_{step}^2$ (mm) |
| latency_total_ms | int | Latensi end-to-end (ms) |
| packet_loss_frame | int | 1 jika frame hilang, 0 jika tidak |
| packet_loss_cumulative | int | Total paket hilang kumulatif |
| is_head_rotating | *tidak dipakai* | Kolom cadangan (selalu 0) |
| lert_triggered | int | 1 jika alert aktif di frame ini |
| lert_text | str | Teks TTS yang diucapkan sistem |
| ground_truth | str | Label arah sebenarnya (dari marker EVENT) |


In [ ]:
def generate_dummy(fname='VNetra_Session_DUMMY.csv', duration_s=60, fps=15):
    """Generate data dummy sesi navigasi VNetra-Lite.

    Menghasilkan file CSV dengan:
    - Baris data frame (skema 18 kolom)
    - Baris EVENT ground truth untuk Confusion Matrix (6 kelas x 10 trial = 60 trial)

    Args:
        fname      : Nama file CSV output.
        duration_s : Durasi sesi dalam detik (min 120s agar semua EVENT masuk).
        fps        : Frame per detik dummy (15 fps default, sesuai StreamService.kt).
    """
    CLASS_LABELS_DUMMY = ['Jam 10', 'Jam 11', 'Jam 12', 'Jam 1', 'Jam 2', 'Jalan Kosong']

    # ── Baris Data Frame ──────────────────────────────────────────────────
    n  = int(duration_s * fps)
    t  = np.linspace(0, duration_s, n)

    # Setiap siklus: idle(10s) → approach(60s) → alert(30s) → recover(20s)
    CYCLE = 120.0
    phase = t % CYCLE
    d_base = np.where(phase < 10,  4500,
             np.where(phase < 70,  4500 - 50 * (phase - 10),
             np.where(phase < 100, 500 + 30  * (phase - 70),
                                   1400 + 21 * (phase - 100))))
    d_obj = np.clip(d_base + np.random.normal(0, 80, n), 300, 5000).astype(int)

    v_raw = np.maximum(0, np.gradient(-d_obj.astype(float), t))
    alpha = 0.15
    v_avg = np.zeros(n)
    for i in range(1, n):
        v_avg[i] = alpha * v_raw[i] + (1 - alpha) * v_avg[i-1]

    a_lin = np.abs(np.gradient(v_avg, t))
    m_buf = np.round(0.5 * a_lin * 0.63**2, 2)
    T_R_const, T_STEP_const = 1.3, 0.63
    ssd   = np.maximum(0, v_avg * (T_R_const + T_STEP_const) - m_buf)
    rv    = 4000.0 - 1000.0
    thresh = (1000 + rv * np.tanh(ssd / rv)).astype(int)
    alert  = (d_obj < thresh).astype(int)

    # Label arah bergantian setiap siklus (simulasi penguji memutar objek)
    direction_cycle = [
        'Arah 10', 'Arah 11', 'Arah 12', 'Arah 1', 'Arah 2', 'aman'
    ]
    cycle_idx = ((t // CYCLE) % 6).astype(int)
    alert_text_arr = np.where(alert, np.array(direction_cycle)[cycle_idx], '')

    lhw  = np.random.randint(2,  45, n)
    lnet = np.random.randint(12, 75, n)
    lal  = np.random.randint(2,  7,  n)
    ltts = np.where(alert, np.random.randint(90, 230, n), 0)
    lbt  = np.random.randint(1,  5,  n)

    import random
    import pandas as pd
    df = pd.DataFrame({
        'timestamp_ms':    (t * 1000).astype(int) + 1700000000000,
        'elapsed_s':       np.round(t, 2),
        'd_obj_mm':        d_obj,
        'v_raw_mmps':      np.round(v_raw, 2),
        'v_avg_mmps':      np.round(v_avg, 2),
        'm_buffer_mm':     m_buf,
        'threshold_T_mm':  thresh,
        'alert_triggered': alert,
        'alert_text':      alert_text_arr,
        'latency_hw_ms':   lhw,   'latency_net_ms':   lnet,
        'latency_algo_ms': lal,   'latency_tts_ms':   ltts,
        'latency_bt_ms':   lbt,   'latency_total_ms': lhw+lnet+lal+ltts+lbt,
        'packet_loss_frame':      np.random.poisson(0.20, n),
        'packet_loss_cumulative': np.random.poisson(0.20, n).cumsum(),
        'is_head_rotating':       (np.random.random(n) < 0.05).astype(int),
    })

    # ── Baris EVENT untuk Confusion Matrix ───────────────────────────────
    # Simulasi 6 kelas x 10 trial = 60 trial
    # Akurasi ~88%: 53 benar, 7 salah (dari 60 total)
    TRIALS_PER_CLASS = 10
    ERROR_RATE       = 0.12          # ~12% trial salah = ~88% akurasi

    direction_to_tts = {
        'Jam 10':      'Arah 10',
        'Jam 11':      'Arah 11',
        'Jam 12':      'Arah 12',
        'Jam 1':       'Arah 1',
        'Jam 2':       'Arah 2',
        'Jalan Kosong':'aman',
    }

    event_lines = []
    ts_base = int(t[0] * 1000) + 1700000000000
    ts_step  = int(duration_s * 1000 / (len(CLASS_LABELS_DUMMY) * TRIALS_PER_CLASS + 1))

    trial_ts = ts_base + ts_step
    for cls in CLASS_LABELS_DUMMY:
        correct_tts = direction_to_tts[cls]
        other_tts   = [v for k, v in direction_to_tts.items() if k != cls]
        for trial in range(TRIALS_PER_CLASS):
            elapsed = (trial_ts - ts_base) / 1000.0
            # Tulis baris EVENT ke dalam CSV
            event_lines.append(
                f'EVENT,{trial_ts},{elapsed:.2f},GROUND_TRUTH,"{cls}"'
            )
            # Patch alert_text di frame dalam jendela [ts-500ms, ts+2000ms]
            t_start_ms = trial_ts - 500
            t_end_ms   = trial_ts + 2000
            is_wrong   = (random.random() < ERROR_RATE)
            tts_label  = random.choice(other_tts) if is_wrong else correct_tts
            mask = (df['timestamp_ms'] >= t_start_ms) & (df['timestamp_ms'] <= t_end_ms)
            if mask.sum() > 0:
                df.loc[mask, 'alert_text']      = tts_label
                df.loc[mask, 'alert_triggered'] = 1

            trial_ts += ts_step

    # ── Simpan ke CSV: data frame + EVENT ────────────────────────────────
    with open(fname, 'w', encoding='utf-8') as fh:
        # Header + baris data
        fh.write(df.to_csv(index=False))
        # Tambahkan baris EVENT di akhir file
        # (_parse_csv_lines() membaca seluruh file, posisi tidak penting)
        fh.write('\n'.join(event_lines) + '\n')

    n_cycles = duration_s / 120.0
    n_events = len(event_lines)
    print(f'Dummy CSV: {fname} | {n} frame | {n_cycles:.1f} siklus | '
          f'Alert: {int(alert.sum())} frame | EVENT markers: {n_events}')
    return df


# Ubah duration_s sesuai kebutuhan. Min ~120s agar EVENT tersebar merata.
df = generate_dummy(duration_s=600)
df.head(3)


---
## Sel 4 — Input Data CSV Sesi Pengujian (Interaktif & Fleksibel)

Sel ini menyediakan antarmuka pemuatan data yang mendukung tiga metode:
1. **Drag & Drop / Tombol Browse:** Klik tombol **Pilih CSV** di bawah atau tarik file CSV langsung ke widget.
2. **Auto-Detect CSV Lapangan:** Jika Anda menyalin file `VNetra_Session_*.csv` ke folder ini, sistem otomatis mendeteksi file lapangan terbaru (mengecualikan file dummy).
3. **Fallback Simulasi Dummy:** Jika belum ada file lapangan, sistem otomatis menggunakan data dummy agar seluruh analisis tetap dapat dipratinjau.

> **Format nama file yang diharapkan:** `VNetra_Session_YYYYMMDD_HHMMSS.csv` (dari folder `Documents/VNetra_Logs/` di Android).


In [ ]:
import io
import os
import glob
import pandas as pd
from IPython.display import display, HTML

# ── 1. Cek Ketersediaan ipywidgets / Google Colab ─────────────────────────
try:
    import ipywidgets as widgets
    HAS_WIDGETS = True
except ImportError:
    HAS_WIDGETS = False

try:
    from google.colab import files as colab_files
    IS_COLAB = True
except ImportError:
    IS_COLAB = False

# ── 2. Logika Seleksi & Sanitasi File CSV ──────────────────────────────────
def get_latest_real_csv():
    """Mencari CSV lapangan asli terbaru (mengecualikan DUMMY dan UPLOADED)."""
    real_csvs = sorted([f for f in glob.glob('VNetra_Session_*.csv') 
                        if 'DUMMY' not in f and 'UPLOADED' not in f])
    return real_csvs[-1] if real_csvs else None

def sanitize_and_prepare_df(df_input):
    """Membersihkan data frame dari baris EVENT non-numerik dan meng-cast tipe data."""
    if 'timestamp_ms' in df_input.columns:
        df_clean = df_input[pd.to_numeric(df_input['timestamp_ms'], errors='coerce').notnull()].copy()
    else:
        df_clean = df_input.copy()

    num_cols = [
        'timestamp_ms', 'elapsed_s', 'd_obj_mm', 'v_raw_mmps', 'v_avg_mmps', 'm_buffer_mm',
        'threshold_T_mm', 'alert_triggered', 'latency_hw_ms', 'latency_net_ms',
        'latency_algo_ms', 'latency_tts_ms', 'latency_bt_ms', 'latency_total_ms',
        'packet_loss_frame', 'packet_loss_cumulative', 'is_head_rotating'
    ]
    for c in num_cols:
        if c in df_clean.columns:
            df_clean[c] = pd.to_numeric(df_clean[c], errors='coerce')

    # Kompatibilitas mundur jika kolom belum ada
    if 'packet_loss_frame' not in df_clean.columns and 'packet_loss_count' in df_clean.columns:
        df_clean['packet_loss_frame'] = df_clean['packet_loss_count'].diff().fillna(0).clip(lower=0).astype(int)
        df_clean['packet_loss_cumulative'] = df_clean['packet_loss_count']
    if 'packet_loss_frame' not in df_clean.columns:
        df_clean['packet_loss_frame'] = 0
        df_clean['packet_loss_cumulative'] = 0
    if 'is_head_rotating' not in df_clean.columns:
        df_clean['is_head_rotating'] = 0

    return df_clean

def load_session_dataframe(uploaded_bytes=None, uploaded_name=None):
    """Memuat DataFrame berdasarkan urutan prioritas: Upload -> File Lapangan Asli -> Dummy."""
    if uploaded_bytes is not None and uploaded_name:
        print(f"[UPLOAD] Memuat file hasil upload: {uploaded_name}")
        with open('VNetra_Session_UPLOADED.csv', 'wb') as f_out:
            f_out.write(uploaded_bytes)
        df_raw = pd.read_csv(io.BytesIO(uploaded_bytes))
    else:
        latest_real = get_latest_real_csv()
        if latest_real:
            print(f"[FILE] Memuat CSV lapangan asli terbaru: {latest_real}")
            df_raw = pd.read_csv(latest_real)
        elif os.path.exists('VNetra_Session_UPLOADED.csv'):
            print("[FILE] Memuat file CSV hasil upload sebelumnya: VNetra_Session_UPLOADED.csv")
            df_raw = pd.read_csv('VNetra_Session_UPLOADED.csv')
        elif os.path.exists('VNetra_Session_DUMMY.csv'):
            print("[INFO] Tidak ada file lapangan asli. Menggunakan data simulasi: VNetra_Session_DUMMY.csv")
            df_raw = pd.read_csv('VNetra_Session_DUMMY.csv')
        else:
            print("[WARN] Menjalankan generator dummy otomatis...")
            df_raw = generate_dummy(duration_s=600)

    df_clean = sanitize_and_prepare_df(df_raw)
    return df_clean

# ── 3. Tampilkan Widget Input (Jika Mendukung) ─────────────────────────────
print("=" * 70)
print("[INPUT DATA SESI PENGUJIAN VNetra-Lite]")
print("=" * 70)

if HAS_WIDGETS:
    upload_widget = widgets.FileUpload(
        accept='.csv',
        multiple=False,
        description='Pilih CSV',
        tooltip='Klik untuk memilih file CSV atau Drag & Drop ke tombol ini',
        button_style='primary',
        icon='upload'
    )
    
    def on_upload_change(change):
        if not upload_widget.value:
            return
        item = upload_widget.value
        # Kompatibilitas ipywidgets v7 dan v8
        if isinstance(item, dict):
            fname = list(item.keys())[0]
            content = item[fname]['content']
        elif isinstance(item, (tuple, list)):
            first = item[0]
            fname = first.get('name', 'uploaded.csv')
            content = first.get('content', b'')
        else:
            return
        global df
        df = load_session_dataframe(uploaded_bytes=content, uploaded_name=fname)
        print(f"[OK] Dataframe berhasil diperbarui ({len(df)} baris | {df['elapsed_s'].max():.1f}s)! Jalankan sel berikutnya.")

    upload_widget.observe(on_upload_change, names='value')
    print(">> Silakan KLIK tombol di bawah atau DRAG & DROP file CSV ke tombol:")
    display(upload_widget)
else:
    print("[INFO] Widget ipywidgets belum terpasang. Menjalankan auto-deteksi file CSV di folder.")
    print("       (Pasang ipywidgets dengan `pip install ipywidgets` untuk tombol upload interaktif)")

# Inisialisasi awal DataFrame
df = load_session_dataframe()
print(f"Dataframe aktif: {len(df)} baris | Durasi: {df['elapsed_s'].max():.1f}s | FPS: {len(df)/df['elapsed_s'].max():.1f}")
df.head(3)


---
## Gambar 4.1 — Perilaku Threshold Adaptif dan Jarak Objek terhadap Waktu

### Tujuan Figur
Figur ini memvalidasi **respons dinamis** sistem. Berbeda dari *static threshold* konvensional (nilai tetap), VNetra-Lite mengadaptasi batas $T$ setiap frame. Figur ini menjawab pertanyaan:

> *"Apakah threshold $T$ benar-benar berubah mengikuti kecepatan gerak, dan apakah selalu menjaga $d_0$ saat sistem diam?"*

### Panduan Membaca Figur
| Elemen Visual | Makna |
|:---|:---|
| Garis biru tebal | Threshold adaptif $T(t)$ — batas peringatan sistem setiap frame |
| Garis oranye | Jarak objek terdekat $d_{obj}(t)$ dari sensor VL53L5CX |
| Simbol segitiga (▲) | Momen alert terpicu — saat $d_{obj} < T$ |
| Garis hijau putus-putus | $d_0 = 1000$ mm — batas aman absolut minimum |
| Garis merah putus-putus | $T_{max} = 4000$ mm — batas jangkauan fisik sensor |

### Interpretasi Hasil
Sistem dikatakan **berhasil** apabila:
1. $T$ naik saat berjalan dan turun saat melambat (adaptif terhadap kecepatan).
2. $T \geq d_0$ sepanjang waktu — tidak pernah turun di bawah zona aman.
3. Alert terpicu *sebelum* $d_{obj}$ menyentuh $d_0$, memberikan jarak reaksi yang cukup.

In [ ]:
fig, ax = plt.subplots(figsize=(7.2, 3.5))

ax.plot(df['elapsed_s'], df['threshold_T_mm'],
        color=C_THRESHOLD, lw=1.8, label='Threshold $T$')
ax.plot(df['elapsed_s'], df['d_obj_mm'],
        color=C_DISTANCE, lw=1.4, alpha=0.85, label='Jarak Objek $d_{obj}$')
ax.axhline(D0,    color=C_SAFE, lw=1.2, ls='--', label='$d_0$')
ax.axhline(T_MAX, color=C_MAX,  lw=1.2, ls=':',  label='$T_{max}$')

alerts = df[df['alert_triggered'] == 1]
if len(alerts) > 0:
    ax.scatter(alerts['elapsed_s'], alerts['d_obj_mm'],
               color=C_ALERT, s=20, zorder=5, marker='^',
               edgecolors='none', label='Alert')

ax.set_xlabel('Waktu (s)')
ax.set_ylabel('Jarak (mm)')
ax.set_title('Threshold Adaptif vs Waktu')
ax.set_xlim(df['elapsed_s'].min(), df['elapsed_s'].max())
ax.set_ylim(0, T_MAX * 1.08)
ax.yaxis.set_major_formatter(ticker.StrMethodFormatter('{x:,.0f}'))
ax.legend(loc='upper right', ncol=3, fontsize=8.5)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
plt.tight_layout()
plt.savefig('output/Fig41_threshold_vs_waktu.png')
plt.show()
print('Saved: output/Fig41_threshold_vs_waktu.png')

---
## Gambar 4.2 — Kurva Kecepatan vs Threshold (Validasi Saturasi *Tanh*)

### Tujuan Figur
Figur ini adalah **instrumen validasi utama formula ambang batas dinamis**.
Setiap titik biru = satu frame pengukuran nyata. Kurva kuning = prediksi teoritis.

### Derivasi Formula (BAB III §3.5.3 – 3.5.5)

**Langkah 1 — Jarak Pengereman** *(Pers. 3.5)*:

$$d_B = v_{avg} \cdot t_{step} - \frac{1}{2}\,|a_{lin}|\cdot t_{step}^2 \tag{3.5}$$

**Langkah 2 — Jarak Reaksi Kognitif** *(Pers. 3.6)*:

$$d_R = v_{avg} \cdot t_R \tag{3.6}$$

**Langkah 3 — Stopping Sight Distance** *(Pers. 3.7 → 3.8)*:

$$SSD = \max(0,\; d_R + d_B) = \max\!\left(0,\; v_{avg}(t_R + t_{step}) - \frac{1}{2}|a_{lin}|t_{step}^2\right) \tag{3.8}$$

**Langkah 4 — Threshold Dinamis Sigmoid** *(Pers. 3.9)*:

$$T = d_0 + (T_{max} - d_0)\cdot\tanh\!\left(\frac{SSD}{T_{max}-d_0}\right) \tag{3.9}$$

| Simbol | Nilai | Keterangan |
|:---:|:---:|:---|
| $t_R$ | 1.3 s | *Perception-Reaction Time* [Kovacs & Nagy, 2020] |
| $t_{step}$ | 0.63 s | Durasi satu langkah [Knoblauch et al.] |
| $d_0$ | 1000 mm | Jarak pelindung minimum |
| $T_{max}$ | 4000 mm | Jangkauan maksimum sensor |
| $m_{buffer}$ | $\frac{1}{2}|a_{lin}|t_{step}^2$ | Jarak pengereman — dilog CSV kolom `m_buffer_mm` |

### Dua Properti Kritis
1. **Diam** ($v_{avg}\to 0$): $d_B, d_R \to 0 \Rightarrow SSD=0 \Rightarrow T = d_0$
2. **Lari** ($v_{avg}\to\infty$): $SSD\to\infty \Rightarrow \tanh\to 1 \Rightarrow T = T_{max}$ (**saturasi aman**)

### Interpretasi
$R^2 \approx 1.0$ membuktikan `NavigationCoordinator.kt` mengimplementasikan Pers. 3.9 dengan benar.


In [ ]:
# ── Fig. 4.2: Kecepatan vs Threshold (Validasi Pers. 3.5–3.9 BAB III) ────────
fig, ax = plt.subplots(figsize=(3.5, 3.5))

# Scatter: setiap titik = 1 frame pengukuran
ax.scatter(df['v_avg_mmps'], df['threshold_T_mm'],
           alpha=0.20, s=5, color=C_SCATTER, edgecolors='none', label='Data per frame')

# ── Percepatan linear per-frame dari gradien kecepatan ───────────────────
# |a_lin| = d(v_avg)/dt  (mm/s²) — sesuai BAB III definisi a_lin
a_lin_per_frame = np.abs(
    np.gradient(df['v_avg_mmps'].values, df['elapsed_s'].values)
)

# ── Per-frame: d_B, d_R, SSD, T_pred (untuk R²) ───────────────────────────
# Pers. 3.5: d_B = v_avg * t_step - 0.5 * |a_lin| * t_step^2
d_B_pred = np.maximum(0, df['v_avg_mmps'] * T_STEP
                        - 0.5 * a_lin_per_frame * T_STEP**2)

# Pers. 3.6: d_R = v_avg * t_R
d_R_pred = df['v_avg_mmps'] * T_R

# Pers. 3.7/3.8: SSD = max(0, d_R + d_B)
ssd_pred = np.maximum(0, d_R_pred + d_B_pred)

# Pers. 3.9: T = d_0 + (T_max - d_0) * tanh(SSD / (T_max - d_0))
rv     = float(T_MAX - D0)
t_pred = D0 + rv * np.tanh(ssd_pred / rv)

# ── Kurva Teoritis (rata-rata |a_lin| sebagai kondisi berjalan normal) ────
a_lin_avg = a_lin_per_frame.mean()          # |a_lin| rata-rata sesi (mm/s²)
v_line    = np.linspace(0, max(df['v_avg_mmps'].max() * 1.08, 1600), 600)

# Pers. 3.5 (teoritis): d_B = v * t_step - 0.5 * a_lin_avg * t_step^2
d_B_line = np.maximum(0, v_line * T_STEP - 0.5 * a_lin_avg * T_STEP**2)

# Pers. 3.6 (teoritis): d_R = v * t_R
d_R_line = v_line * T_R

# Pers. 3.7/3.8 (teoritis): SSD = max(0, d_R + d_B)
ssd_line = np.maximum(0, d_R_line + d_B_line)

# Pers. 3.9 (teoritis): T = d_0 + rv * tanh(SSD / rv)
t_theory = D0 + rv * np.tanh(ssd_line / rv)

ax.plot(v_line, t_theory, color=C_THEORY, lw=2.0,
        label=rf'Teori Pers. 3.9 ($\bar{{a}}$={a_lin_avg:.0f} mm/s²)')
ax.axhline(D0,    color=C_SAFE, lw=1.0, ls='--', label=f'$d_0$ = {D0:,} mm')
ax.axhline(T_MAX, color=C_MAX,  lw=1.0, ls=':',  label=f'$T_{{max}}$ = {T_MAX:,} mm')

# ── R² ─────────────────────────────────────────────────────────────────────
ss_res = np.sum((df['threshold_T_mm'] - t_pred) ** 2)
ss_tot = np.sum((df['threshold_T_mm'] - df['threshold_T_mm'].mean()) ** 2)
r2 = 1 - ss_res / ss_tot if ss_tot > 0 else float('nan')

ax.text(0.97, 0.05, f'$R^2 = {r2:.4f}$',
        transform=ax.transAxes, ha='right', va='bottom', fontsize=9,
        bbox=dict(boxstyle='round,pad=0.3', facecolor='white', edgecolor='#AAAAAA'))

ax.set_xlabel('$v_{avg}$ (mm/s)')
ax.set_ylabel('$T$ (mm)')
ax.set_title('Kecepatan vs Threshold', fontsize=9)
ax.set_ylim(D0 * 0.9, T_MAX * 1.04)
ax.yaxis.set_major_formatter(ticker.StrMethodFormatter('{x:,.0f}'))
ax.legend(fontsize=7.5)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
plt.tight_layout()
outpath = os.path.join(OUTPUT_DIR, 'Fig42_velocity_vs_threshold.png')
plt.savefig(outpath, dpi=300, bbox_inches='tight')
plt.show()
print(f'Saved: {outpath} | R2={r2:.4f}')
print(f'  a_lin rata-rata={a_lin_avg:.1f} mm/s² | d_B @1000mm/s = {max(0, 1000*T_STEP - 0.5*a_lin_avg*T_STEP**2):.0f} mm')
print(f'  d_R @1000mm/s = {1000*T_R:.0f} mm | SSD @1000mm/s = {max(0, 1000*T_STEP - 0.5*a_lin_avg*T_STEP**2 + 1000*T_R):.0f} mm')


---
## Gambar 4.3 — Dekomposisi Latensi *End-to-End*

### Tujuan Figur
Sistem ETA efektif mensyaratkan latensi total $< t_R = 1300$ ms (batas reaksi kognitif manusia [44]). Figur ini memperlihatkan kontribusi setiap komponen.

### Komponen Latensi
| Komponen | Kolom CSV | Penjelasan |
|:---|:---|:---|
| **Hardware** | `latency_hw_ms` | Waktu sensor VL53L5CX + pemrosesan ESP32 |
| **Jaringan IoT** | `latency_net_ms` | RTT/2 paket UDP WiFi ESP32 → Android |
| **Algoritma** | `latency_algo_ms` | EWMA + Mahony AHRS + komputasi Tanh |
| **TTS Audio** | `latency_tts_ms` | Render + buffer Android TextToSpeech |
| **Bluetooth** | `latency_bt_ms` | Transmisi audio ke headset BT |

### Panduan Membaca Figur
- **Panel kiri (stacked bar):** Distribusi latensi per segmen waktu. Garis merah = batas $t_R = 1300$ ms.
- **Panel kanan (box plot):** Sebaran statistik per komponen — identifikasi komponen paling tidak stabil.
- Jika batang secara konsisten melebihi garis merah, sistem perlu dioptimasi (kompresi payload UDP, pengurangan teks TTS).

In [ ]:
fig, (ax_bar, ax_box) = plt.subplots(1, 2, figsize=(7.2, 3.5),
                                      gridspec_kw={'width_ratios': [2, 1]})

df2 = df.copy()
df2['time_bin'] = pd.cut(df2['elapsed_s'], bins=15)
grp = df2.groupby('time_bin', observed=True)[[
    'latency_hw_ms','latency_net_ms','latency_algo_ms','latency_tts_ms','latency_bt_ms'
]].mean()
x_ticks  = range(len(grp))
x_labels = [f'{i.left:.0f}' for i in grp.index]
lat_cols = ['latency_hw_ms','latency_net_ms','latency_algo_ms','latency_tts_ms','latency_bt_ms']
lat_lbl  = ['HW','Jaringan','Algo','TTS','BT']
lat_clr  = ['#0072B2','#009E73','#E69F00','#D55E00','#CC79A7']
bottom   = np.zeros(len(grp))
for col, lbl, clr in zip(lat_cols, lat_lbl, lat_clr):
    v = grp[col].fillna(0).values
    ax_bar.bar(x_ticks, v, bottom=bottom, label=lbl, color=clr, alpha=0.85, width=0.72)
    bottom += v
ax_bar.axhline(800, color='#0072B2', lw=1.2, ls=':', label='Target E2E (800 ms)')
ax_bar.axhline(T_R*1000, color=C_MAX, lw=1.2, ls='--', label=f'Batas Kognitif $t_R$ ({T_R*1000:.0f} ms)')
ax_bar.set_xticks(x_ticks)
ax_bar.set_xticklabels(x_labels, rotation=45, fontsize=7)
ax_bar.set_xlabel('Waktu (s)')
ax_bar.set_ylabel('Latensi (ms)')
ax_bar.set_title('(a) Stacked per Segmen')
ax_bar.legend(fontsize=7.5, loc='upper left', ncol=2)
ax_bar.spines['top'].set_visible(False)
ax_bar.spines['right'].set_visible(False)

box_data = [df[c].values for c in lat_cols]
bp = ax_box.boxplot(box_data, patch_artist=True,
                    medianprops=dict(color='black', lw=1.5),
                    whiskerprops=dict(color='#555555', lw=0.8),
                    capprops=dict(color='#555555', lw=0.8),
                    flierprops=dict(marker='.', color='#888888', alpha=0.4, ms=2))
for patch, clr in zip(bp['boxes'], lat_clr):
    patch.set_facecolor(clr); patch.set_alpha(0.6)
ax_box.set_xticks(range(1, 6))
ax_box.set_xticklabels(lat_lbl, fontsize=8)
ax_box.set_ylabel('Latensi (ms)')
ax_box.set_title('(b) Distribusi')
ax_box.axhline(800, color='#0072B2', lw=1.0, ls=':')
ax_box.axhline(T_R*1000, color=C_MAX, lw=1.0, ls='--')
ax_box.spines['top'].set_visible(False)
ax_box.spines['right'].set_visible(False)

ax_bar.set_ylim(0, 600)
ax_box.set_ylim(0, 600)
fig.suptitle('Dekomposisi Latensi End-to-End', fontsize=11, fontweight='bold')
plt.tight_layout()
plt.savefig('output/Fig43_latency_decomposition.png')
lat_mean = df['latency_total_ms'].mean()
plt.show()
print(f'Saved: output/Fig43_latency_decomposition.png | Mean latency: {lat_mean:.1f} ms')

---
## Gambar 4.4 — Reliabilitas Jaringan: Packet Loss dan PDR

### Tujuan Figur
VNetra-Lite menggunakan **UDP** (*connectionless*) — setiap paket yang hilang = satu frame jarak yang tidak diproses Android. Figur ini mengukur:
1. **Pola temporal packet loss** — acak (noise) atau terkonsentrasi (interferensi lokal)?
2. **Packet Delivery Ratio (PDR)** — rasio paket berhasil diterima. Standar minimum: **PDR ≥ 95%** [53].

### Interpretasi
| Pola Kurva | Diagnosis | Tindakan |
|:---|:---|:---|
| Naik landai merata | Noise background normal | Acceptable |
| Loncat tiba-tiba | Interferensi / roaming WiFi | Optimasi kanal |
| PDR ≥ 95% | Memenuhi standar | Laporkan sebagai valid |
| PDR < 95% | Di bawah standar | Pertimbangkan redundansi paket |

In [ ]:
fig, (ax_cum, ax_gauge) = plt.subplots(1, 2, figsize=(7.2, 3.0),
                                        gridspec_kw={'width_ratios': [3, 1]})

# Panel kiri: tampilkan packet_loss_cumulative langsung (sudah kumulatif dari kode Android)
cum_loss = df['packet_loss_cumulative']
ax_cum.fill_between(df['elapsed_s'], cum_loss, alpha=0.12, color=C_MAX)
ax_cum.plot(df['elapsed_s'], cum_loss, color=C_MAX, lw=1.5, label='Packet Loss Kumulatif')
ax_cum.set_xlabel('Waktu (s)')
ax_cum.set_ylabel('Paket Hilang Kumulatif')
ax_cum.set_title('(a) Packet Loss vs Waktu')
ax_cum.set_xlim(df['elapsed_s'].min(), df['elapsed_s'].max())
ax_cum.legend(fontsize=9)
ax_cum.spines['top'].set_visible(False)
ax_cum.spines['right'].set_visible(False)

# Panel kanan: PDR dari packet_loss_frame (per-frame, BUKAN kumulatif)
total_lost   = int(df['packet_loss_frame'].sum())   # total paket hilang selama sesi
total_frames = len(df)
pdr = (1 - total_lost / (total_frames + total_lost)) * 100
bar_color = C_SAFE if pdr >= 95 else (C_THEORY if pdr >= 90 else C_MAX)

ax_gauge.barh(['PDR'], [100], color='#EEEEEE', height=0.5, edgecolor='#BBBBBB', lw=0.5)
ax_gauge.barh(['PDR'], [pdr], color=bar_color, height=0.5)
ax_gauge.axvline(95, color='#333333', lw=1.0, ls='--')
ax_gauge.text(pdr/2, 0, f'{pdr:.1f}%', color='white', fontsize=11,
              fontweight='bold', ha='center', va='center')
ax_gauge.set_xlim(0, 100)
ax_gauge.set_xlabel('PDR (%)')
ax_gauge.set_title('(b) PDR')
ax_gauge.tick_params(axis='y', left=False, labelleft=False)
ax_gauge.spines['top'].set_visible(False)
ax_gauge.spines['right'].set_visible(False)
ax_gauge.spines['left'].set_visible(False)

fig.suptitle('Packet Loss dan Packet Delivery Ratio (PDR)', fontsize=11, fontweight='bold')
plt.tight_layout()
plt.savefig('output/Fig44_packet_loss_pdr.png')
plt.show()
status = 'OK' if pdr >= 95 else 'PERLU INVESTIGASI'
loss_rate_pct = (total_lost / total_frames) * 100
print(f'Saved: output/Fig44_packet_loss_pdr.png | PDR = {pdr:.2f}% [{status}]')
print(f'Total paket hilang: {total_lost} paket | Loss rate: {loss_rate_pct:.3f}% per frame')

---
## Tabel 4.x — Ringkasan Statistik Pengujian

Tabel ini merangkum metrik kuantitatif utama sesi pengujian untuk **Bab 4 (Hasil dan Pembahasan)**.

| Metrik | Kolom Sumber | Satuan |
|:---|:---|:---:|
| FPS rata-rata | lapsed_s | fps |
| Latensi rata-rata | latency_total_ms | ms |
| Latensi std. dev | latency_total_ms | ms |
| Latensi median | latency_total_ms | ms |
| Latensi maksimum | latency_total_ms | ms |
| Packet Delivery Ratio | packet_loss_frame | % |

> **Panduan pelaporan:** Laporkan latensi sebagai Mean ± SD, dan bandingkan PDR dengan standar minimum **PDR ≥ 95%** [53, Pers. 2.13].


In [ ]:
total_lost = int(df['packet_loss_frame'].sum())   # total paket hilang (dari kolom per-frame)
pdr      = (1 - total_lost / (len(df) + total_lost)) * 100
lat_mean = df['latency_total_ms'].mean()
lat_std  = df['latency_total_ms'].std()
lat_med  = df['latency_total_ms'].median()
lat_max  = df['latency_total_ms'].max()
fps_avg  = len(df) / df['elapsed_s'].max()
status_lat = 'OK -- Memenuhi' if lat_mean < T_R*1000 else 'PERLU OPTIMASI'
status_pdr = 'OK' if pdr >= 95 else 'PERLU INVESTIGASI'

print('=' * 62)
print('   TABEL RINGKASAN STATISTIK SESI PENGUJIAN VNetra-Lite')
print('=' * 62)
print(f'  Durasi Sesi                  : {df["elapsed_s"].max():.1f} detik')
print(f'  Total Frame                  : {len(df):,} frame @ {fps_avg:.1f} fps')
print('-' * 62)
print('  [THRESHOLD ADAPTIF]')
print(f'  Threshold Min/Max            : {df["threshold_T_mm"].min():,} / {df["threshold_T_mm"].max():,} mm')
print(f'  Threshold Rata-rata          : {df["threshold_T_mm"].mean():.0f} mm')
print(f'  Kecepatan v_avg Maks         : {df["v_avg_mmps"].max():.0f} mm/s = {df["v_avg_mmps"].max()/1000:.2f} m/s')
print(f'  M_buffer Rata-rata           : {df["m_buffer_mm"].mean():.1f} mm')
print('-' * 62)
print('  [PERINGATAN (ALERT)]')
print(f'  Total Frame Peringatan       : {df["alert_triggered"].sum()} frame')
print(f'  Rasio Frame Peringatan       : {df["alert_triggered"].mean()*100:.1f}%')
print('-' * 62)
print('  [LATENSI END-TO-END]')
print(f'  Latensi Total Mean +/- SD    : {lat_mean:.1f} +/- {lat_std:.1f} ms')
print(f'  Latensi Total Median         : {lat_med:.1f} ms')
print(f'  Latensi Total Maks           : {lat_max} ms')
print(f'  Latensi Jaringan Mean        : {df["latency_net_ms"].mean():.1f} ms')
print(f'  Batas Kognitif t_R           : {T_R*1000:.0f} ms  [Kovacs & Nagy, 2020]')
print(f'  Status                       : {status_lat}')
print('-' * 62)
print('  [RELIABILITAS JARINGAN]')
print(f'  Total Packet Loss            : {total_lost} paket')
print(f'  Packet Delivery Ratio (PDR)  : {pdr:.2f}%')
print(f'  Status (>= 95%?)             : {status_pdr}')
print('=' * 62)

---
## Gambar 4.5 — *Confusion Matrix* Akurasi Deteksi Arah

### Tujuan Figur
Figur ini mengukur **akurasi spasial** sistem dalam mendeteksi arah rintangan.
Setiap baris = arah yang *sebenarnya* (ground truth dari marker EVENT).
Setiap kolom = arah yang *diprediksi* oleh sistem (teks TTS yang diucapkan).

### Prasyarat
CSV harus mengandung baris EVENT (marker ground truth). Penguji menekan tombol
di `StreamActivity` sebelum setiap trial sehingga `logTestMarker()` menulis:
```
EVENT,<timestamp_ms>,<elapsed_s>,GROUND_TRUTH,"Jam 12"
```

### Kolom yang Digunakan
| Kolom | Keterangan |
|:---|:---|
| `event_ts_ms` | Timestamp marker dari penguji (milidetik) |
| `ground_truth` | Label arah sebenarnya: "Jam 10" .. "Jam 2" atau "Jalan Kosong" |
| `alert_text` | Teks yang diucapkan TTS oleh sistem (prediksi) |

### Interpretasi
- **Diagonal tinggi** = sistem akurat.
- **Off-diagonal** = tipe kesalahan (misal Jam 11 terdeteksi sebagai Jam 12).
- **Target lulus:** akurasi rata-rata $\geq 85\%$ (BAB III, Pengujian 3.7.2 [Pers. 2.12]).

> **Jika tidak ada baris EVENT di CSV:** sel ini menampilkan confusion matrix
> dari data dummy acak — nilai tidak representatif.


In [ ]:
# ═══════════════════════════════════════════════════════════════════
# Fig. 4.5 — Confusion Matrix Akurasi Deteksi Arah
# ═══════════════════════════════════════════════════════════════════
from io import StringIO
from matplotlib.colors import LinearSegmentedColormap

CLASS_LABELS             = ['Jam 10', 'Jam 11', 'Jam 12', 'Jam 1', 'Jam 2', 'Jalan Kosong']
PREDICTION_WINDOW_START_S = -1.0  # mundur 1 detik (toleransi keterlambatan tekan tombol)
PREDICTION_WINDOW_END_S   =  2.0  # maju 2 detik setelah marker ditekan

# Kolom numerik yang wajib di-cast — mencakup skema baru (18 kolom)
NUMERIC_COLS = [
    'v_avg_mmps', 'm_buffer_mm', 'd_obj_mm', 'latency_total_ms', 'threshold_T_mm',
    'packet_loss_frame', 'packet_loss_cumulative', 'is_head_rotating',  # kolom baru
]


def _parse_csv_lines(csv_path):
    """Memisahkan baris EVENT (marker) dari baris data biasa.

    Returns:
        raw_lines  : list[str] baris data (termasuk header) untuk pd.read_csv
        event_rows : list[dict] dengan 'event_ts_ms' dan 'ground_truth'
    """
    raw_lines, event_rows = [], []
    header_written = False
    with open(csv_path, 'r', encoding='utf-8') as fh:
        for line in fh:
            line = line.strip()
            if not line:
                continue
            if line.startswith('EVENT,'):
                parts = line.split(',', 5)
                event_rows.append({
                    'event_ts_ms': int(parts[1]),
                    'ground_truth': parts[4].strip('"'),
                })
            elif 'timestamp_ms' in line:
                if not header_written:          # tulis header hanya sekali
                    raw_lines.append(line)
                    header_written = True
            else:
                raw_lines.append(line)
    return raw_lines, event_rows


def _cast_numeric_columns(df_input):
    """Memaksa kolom yang diketahui ke tipe numerik.

    Mencegah TypeError diam-diam saat kolom bertipe object karena
    artifact parsing CSV (spasi ekstra, baris header ganda, dsb).
    """
    for col in NUMERIC_COLS:
        if col in df_input.columns:
            df_input[col] = pd.to_numeric(df_input[col], errors='coerce')
    return df_input


def load_csv_with_markers(csv_path):
    """Membaca CSV sesi VNetra yang mengandung baris EVENT.

    Args:
        csv_path: path ke file CSV sesi (mengandung baris data + baris EVENT).

    Returns:
        df_data    : DataFrame baris data biasa (skema 18 kolom)
        df_markers : DataFrame ground-truth markers (event_ts_ms, ground_truth)
    """
    raw_lines, event_rows = _parse_csv_lines(csv_path)
    df_data = pd.read_csv(StringIO('\n'.join(raw_lines)))
    df_data = _cast_numeric_columns(df_data)
    return df_data, pd.DataFrame(event_rows)


def _normalize_direction_label(alert_text):
    """Menormalisasi teks alert TTS menjadi salah satu dari CLASS_LABELS."""
    text = str(alert_text).strip().lower()
    if not text or text in ('nan', '', '0'):
        return 'Jalan Kosong'
    direction_map = [
        ('10', 'Jam 10'), ('11', 'Jam 11'), ('12', 'Jam 12'),
        ('jam 1', 'Jam 1'), ('jam 2', 'Jam 2'),
        (' 1', 'Jam 1'), (' 2', 'Jam 2'),
        ('kosong', 'Jalan Kosong'), ('aman', 'Jalan Kosong'),
    ]
    for keyword, label in direction_map:
        if keyword in text:
            return label
    return 'Jalan Kosong'


def extract_predictions(df_data, df_markers):
    """Mencocokkan setiap ground-truth marker dengan modus prediksi sistem.

    Untuk setiap marker, ambil semua frame dalam jendela waktu
    [start-1s, start+2s] dan pilih label terbanyak (modus).
    """
    results = []
    for _, marker in df_markers.iterrows():
        t_start = marker['event_ts_ms'] + PREDICTION_WINDOW_START_S * 1000
        t_end   = marker['event_ts_ms'] + PREDICTION_WINDOW_END_S   * 1000
        window  = df_data[
            (df_data['timestamp_ms'] >= t_start) &
            (df_data['timestamp_ms'] <= t_end)
        ]
        if window.empty:
            print(f"[SKIP] Tidak ada frame dalam jendela GT='{marker['ground_truth']}'")
            continue
        predicted = window['alert_text'].apply(_normalize_direction_label).mode().iloc[0]
        results.append({'ground_truth': marker['ground_truth'], 'predicted': predicted})
    return results


# ── Resolve csv_path: pakai CSV terbaru yang ditemukan di folder ──────────
# Strategi: cari VNetra_Session_*.csv di folder yang sama dengan notebook.
# Jika tidak ada, gunakan DUMMY CSV yang dibuat di Sel 3.
# ── Resolve csv_path: cari file upload -> lapangan asli -> fallback dummy ──
real_csvs = sorted([f for f in glob.glob('VNetra_Session_*.csv') if 'DUMMY' not in f and 'UPLOADED' not in f])
if os.path.exists('VNetra_Session_UPLOADED.csv'):
    csv_path = 'VNetra_Session_UPLOADED.csv'
    print(f'CSV Aktif: Menggunakan file hasil upload: {csv_path}')
elif real_csvs:
    csv_path = real_csvs[-1]
    print(f'CSV Aktif: Menggunakan CSV lapangan asli terbaru: {csv_path}')
else:
    csv_path = 'VNetra_Session_DUMMY.csv'
    print(f'CSV Aktif: Tidak ada CSV lapangan. Menggunakan dummy: {csv_path}')

# ── Jalankan pipeline confusion matrix ────────────────────────────────────
df_data, df_markers = load_csv_with_markers(csv_path)
print(f'Frame data: {len(df_data)} | Ground truth markers: {len(df_markers)}')
results = extract_predictions(df_data, df_markers)

# ── Hitung confusion matrix ───────────────────────────────────────────────
n_classes = len(CLASS_LABELS)
label_to_index = {label: i for i, label in enumerate(CLASS_LABELS)}
cm = np.zeros((n_classes, n_classes), dtype=int)
for result in results:
    gt_idx   = label_to_index.get(result['ground_truth'], -1)
    pred_idx = label_to_index.get(result['predicted'],    -1)
    if gt_idx >= 0 and pred_idx >= 0:
        cm[gt_idx][pred_idx] += 1

accuracy = np.trace(cm) / cm.sum() if cm.sum() > 0 else 0.0
print(f'Akurasi keseluruhan: {accuracy*100:.2f}% (N={cm.sum()})')


---
## Gambar 4.5 — Heatmap *Confusion Matrix* Akurasi Deteksi Arah

Sel ini menghasilkan **Fig. 4.5**: heatmap confusion matrix dengan anotasi nilai numerik di setiap sel.
- Warna biru makin tua = frekuensi deteksi makin tinggi
- **Diagonal** = prediksi benar (*True Positive*)
- **Off-diagonal** = kesalahan klasifikasi (misal: Jam 11 terdeteksi sebagai Jam 12)

> Jalankan **setelah** Sel Confusion Matrix (Sel 19) — variabel `cm`, `n_classes`, `accuracy` harus sudah terdefinisi.


In [ ]:
# Fig. 4.5 — Heatmap Confusion Matrix (YOLO-style)
import numpy as _np_cm
from matplotlib.colors import LinearSegmentedColormap as _LSC

fig, ax = plt.subplots(figsize=(8, 7))
fig.patch.set_facecolor('white')
ax.set_facecolor('white')

# Colormap: 0 -> putih, non-zero -> biru bertingkat (seperti YOLO)
cmap_yolo = _LSC.from_list(
    'yolo_cm',
    [(1, 1, 1), '#BDD7EE', '#2E75B6', '#1F3864'],
    N=256
)
cmap_yolo.set_bad(color='white')  # sel bernilai 0 = putih bersih

# Mask nilai 0 agar background putih tampak (bukan warna min colormap)
cm_masked = _np_cm.ma.masked_where(cm == 0, cm.astype(float))

im = ax.imshow(cm_masked, cmap=cmap_yolo, aspect='equal',
               vmin=0, vmax=max(1, cm.max()))

# Colorbar
cb = plt.colorbar(im, ax=ax, fraction=0.040, pad=0.03)
cb.ax.tick_params(labelsize=8)
cb.outline.set_linewidth(0.5)

# Ticks & Labels
ax.set_xticks(range(n_classes))
ax.set_yticks(range(n_classes))
ax.set_xticklabels(CLASS_LABELS, rotation=45, ha='right', fontsize=8.5)
ax.set_yticklabels(CLASS_LABELS, fontsize=8.5)

# Sumbu X = Prediksi Sistem, Sumbu Y = Kondisi Sebenarnya (Ground Truth)
ax.set_xlabel('Prediksi Sistem (Predicted)', fontsize=10, labelpad=8)
ax.set_ylabel('Kondisi Aktual (Ground Truth)', fontsize=10, labelpad=8)
ax.set_title(f'Confusion Matrix Deteksi Arah (Akurasi: {accuracy*100:.1f}%)', fontsize=11, pad=12, fontweight='bold')

# Anotasi: tampilkan hanya nilai > 0 (YOLO: sel 0 = kosong)
for i in range(n_classes):
    for j in range(n_classes):
        val = cm[i, j]
        if val == 0:
            continue  # biarkan putih
        brightness = val / max(1, cm.max())
        clr = 'white' if brightness > 0.55 else '#1A1A2E'
        ax.text(j, i, str(val), ha='center', va='center',
                fontsize=9, color=clr, fontweight='bold')

# Border tipis (YOLO style)
for spine in ax.spines.values():
    spine.set_linewidth(0.6)
    spine.set_color('#CCCCCC')

# Grid samar antar sel
import numpy as _np2
ax.set_xticks(_np2.arange(-0.5, n_classes, 1), minor=True)
ax.set_yticks(_np2.arange(-0.5, n_classes, 1), minor=True)
ax.grid(which='minor', color='#E8E8E8', linewidth=0.5)
ax.tick_params(which='minor', length=0)

plt.tight_layout()
outpath = os.path.join(OUTPUT_DIR, 'Fig45_confusion_matrix.png')
plt.savefig(outpath, dpi=300, bbox_inches='tight', facecolor='white')
plt.show()
print(f'Saved: {outpath}')
print(f'Akurasi={accuracy*100:.1f}% | N={cm.sum()}')


---
## Tabel 4.y — Metrik Per Kelas: *Precision*, *Recall*, dan *F1-Score*

Tabel ini diturunkan dari **Gambar 4.5** (*Confusion Matrix*) dan menyajikan metrik
evaluasi per kelas arah secara numerik untuk penulisan **Bab 4**.

| Metrik | Formula | Makna |
|:---|:---|:---|
| **Precision** | $TP / (TP + FP)$ | Dari semua prediksi kelas X, berapa yang benar? |
| **Recall** | $TP / (TP + FN)$ | Dari semua ground truth kelas X, berapa yang terdeteksi? |
| **F1-Score** | $2 \cdot P \cdot R / (P + R)$ | Harmonic mean Precision & Recall |

> **Target lulus:** F1-Score rata-rata $\geq 85\%$ (BAB III Pengujian 3.7.2, Pers. 2.12).


In [ ]:
# Tabel 4.y — Metrik per Kelas (Precision / Recall / F1-Score)
import pandas as pd

rows = []
for i, lbl in enumerate(CLASS_LABELS):
    tp = int(cm[i, i])
    fp = int(cm[:, i].sum()) - tp
    fn = int(cm[i, :].sum()) - tp
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    recall    = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    f1        = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0.0
    support   = int(cm[i, :].sum())
    rows.append({
        'Kelas Arah':    lbl,
        'Precision (%)': f'{precision*100:.1f}',
        'Recall (%)':    f'{recall*100:.1f}',
        'F1-Score (%)':  f'{f1*100:.1f}',
        'Support (N)':   support,
    })

df_metrics = pd.DataFrame(rows)

# Hitung rata-rata makro
avg_p  = sum(float(r['Precision (%)']) for r in rows) / len(rows)
avg_r  = sum(float(r['Recall (%)'])    for r in rows) / len(rows)
avg_f1 = sum(float(r['F1-Score (%)'])  for r in rows) / len(rows)
df_metrics.loc[len(df_metrics)] = {
    'Kelas Arah':    'Rata-rata Makro',
    'Precision (%)': f'{avg_p:.1f}',
    'Recall (%)':    f'{avg_r:.1f}',
    'F1-Score (%)':  f'{avg_f1:.1f}',
    'Support (N)':   int(cm.sum()),
}

# Tampilkan sebagai DataFrame terformat
display(df_metrics.style
    .set_caption(f'Tabel 4.y: Metrik Evaluasi per Kelas Arah (Akurasi Keseluruhan = {accuracy*100:.1f}%)')
    .set_properties(**{'text-align': 'center'})
    .set_table_styles([{'selector': 'th', 'props': [('text-align', 'center')]}])
    .apply(lambda x: ['background-color: #EBF3FB' if x.name == len(df_metrics)-1
                       else '' for _ in x], axis=1)
)

# Ekspor ke CSV untuk pelaporan
df_metrics.to_csv(os.path.join(OUTPUT_DIR, 'Tabel4y_metrik_per_kelas.csv'), index=False)
status = 'LULUS' if avg_f1 >= 85.0 else 'PERLU OPTIMASI'
print(f'Saved: output/Tabel4y_metrik_per_kelas.csv')
print(f'Status: F1 rata-rata = {avg_f1:.1f}% -> {status} (target >= 85%)')
